# 1 · Structured extraction with GPT-4 Turbo

Reads each full text (PDF) of the corpus and asks GPT-4 Turbo (`gpt-4-turbo-2024-04-09`, temperature 0) to return a structured record with seven fields: main research question, methods and experiments, results, discussion, conclusions, future work and proposed keywords.

* **Input:** PDFs in `../data` (files 001–125) and `../data2` (files 126–146). Full texts are not distributed with this repository; retrieve them through the DOIs in `data/corpus_manifest.csv`.
* **Output:** `df_combined2.pkl`, one record per document. The published version of these records is `data/extraction_records.csv`.
* **Article:** Methods, representation of each record. The records were not verified manually (see the article for the rationale).
* **Requirements:** an OpenAI API key (replace `<Your API Key>`), `langchain-community`, `pypdf`, `openai`, `pandas`.

# FSO literature review

Version: 01
- Reading the files
- Creating the summary dataframe

### Preamble

## Imports and API key

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI


In [ ]:
os.environ["OPENAI_API_KEY"] = "<Your API Key>"

## Input folders

In [ ]:
# Directories
directory_path = Path('../data')    # 125 papers
directory_path2 = Path('../data2')  # 21 papers

### Open AI Client Initialization

## Model settings
Model `gpt-4-turbo-2024-04-09`, temperature 0, no frequency or presence penalty.

In [ ]:
# Initialize the OpenAI client
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # This is the default and can be omitted
)
model="gpt-4-turbo-2024-04-09"
frequency_penalty = 0
presence_penalty= 0
temperature = 0

## PDF reader
Extracts the full text of one PDF with LangChain's `PyPDFLoader`.

In [ ]:
def read_single_pdf(pdf_path: str) -> str:
    """
    Reads a single PDF file using LangChain's PyPDFLoader and returns its text.

    :param pdf_path: Path to the PDF file.
    :return: Extracted text from the PDF.
    """
    pdf_file = Path(pdf_path)

    # Verificar si el archivo existe y es un PDF
    if not pdf_file.exists() or pdf_file.suffix.lower() != ".pdf":
        raise ValueError(f"Invalid file: '{pdf_path}'. Ensure it is a PDF and exists.")

    try:
        loader = PyPDFLoader(str(pdf_file))
        pages = loader.load()  # Cargar todas las páginas
        text = "\n".join([page.page_content for page in pages])  # Unir páginas en un solo texto
        return text
    except Exception as e:
        print(f"Error reading {pdf_file.name}: {e}")
        return ""  # Retorna cadena vacía si hay error

In [ ]:
# Testing the function...
# Check if the directory exists
if not directory_path2.exists() or not directory_path2.is_dir():
    raise ValueError(f"Directory '{directory_path2}' does not exist or is not a valid directory.")

# List all PDF files in the directory
pdf_files = sorted(directory_path2.glob("*.pdf"), key=lambda f: int(f.stem))
print(pdf_files[:10])

### File Processing 

## Extraction prompt
Same text as `prompts/structured_extraction_prompt_gpt4turbo.txt`. `{paper}` is replaced by the full text.

In [ ]:
prompt_template = """
CONTEXT: This task involves summarizing scientific papers related to Free-Space Optical (FSO) communications in a structured and academically rigorous manner. Your objective is to extract and present key information in a Python dictionary, ensuring clarity, precision, and adherence to scholarly language.
The extracted summaries must be comprehensive, information-dense, and faithful to the original document, capturing essential concepts, methodologies, findings, and contributions in the field of FSO communications.
Your response must strictly follow the specified Python dictionary format, preserving the exact wording for the Title and Abstract while systematically structuring all extracted elements.
Ensure that each summary reflects the technical depth, research significance, and scientific accuracy required in academic discourse on FSO technologies, security, reliability, and atmospheric challenges.

TONE: Your interaction with users is professional, guiding them in structuring and writing effective literature reviews.
Maintain an academic and formal tone, mirroring the style of peer-reviewed publications.
Ensure objectivity, clarity, and a focus on evidence-based insights.

PROCEDURE:
1.	Carefully read the provided text to extract key information.
2.	Extract the Title and the Abstract exactly as they appear in the document, ensuring absolute fidelity to the original text.
3.	Extract the Main Research Question from the document.
4.	Extract the Methods and Experiments described in the document, detailing how the study was conducted.
5.	Extract the Results reported in the study, including key findings and observations.
6.	Extract the Discussion, highlighting the interpretation of results, comparisons with previous work, and the significance of findings.
7.	Extract the Conclusions drawn by the authors, summarizing the main takeaways of the research.
8.	Extract the Future Work proposed by the authors, identifying areas for further research.
9.	Propose a list of five (5) “Proposed Keywords” that best represent the content of the document, ensuring they reflect its core themes and contributions.

OUTPUT FORMAT:
Return your response in a Python dictionary following the next example:
{{
    "Title": "Security Challenges and Solutions in Free-Space Optical Communications",
    "Abstract": "This paper investigates the security vulnerabilities of Free-Space Optical (FSO) communication systems in atmospheric turbulence conditions. The study examines potential eavesdropping risks and evaluates the effectiveness of physical-layer security techniques. The researchers propose a hybrid cryptographic-physical layer approach to mitigate interception risks, conducting experiments in controlled turbulence environments to assess performance. The results demonstrate that integrating adaptive beamforming with key distribution enhances security, reducing the probability of successful eavesdropping by 78%.",
    "Main Research Question": "How can Free-Space Optical (FSO) communication security be improved to mitigate eavesdropping risks under atmospheric turbulence conditions?",
    "Methods and Experiments": "The study employs a combination of physical-layer security techniques and cryptographic protocols. Controlled experiments were conducted in a turbulence chamber to evaluate the interception probability under varying turbulence intensities. The proposed security framework integrates adaptive beamforming with cryptographic key exchange.",
    "Results": "Experimental findings indicate that adaptive beamforming combined with key distribution reduces the probability of successful eavesdropping by 78%. The results highlight the importance of multi-layered security approaches in FSO systems.",
    "Discussion": "The study demonstrates that physical-layer security techniques alone are insufficient under strong turbulence conditions. The integration of cryptographic measures significantly enhances the resilience of FSO systems against eavesdropping attacks. The paper discusses the trade-offs between security and system complexity.",
    "Conclusions": "A hybrid approach combining adaptive beamforming with cryptographic techniques effectively enhances the security of FSO communications. The study suggests that future research should explore AI-driven beam adaptation to further improve security in dynamic environments.",
    "Future Work": "Future research should focus on developing AI-driven adaptive beamforming techniques to further enhance security under varying atmospheric conditions. Additionally, investigating quantum cryptography integration with FSO systems could provide enhanced security guarantees.",
    "Proposed Keywords": ["Free-Space Optical Communications", "Physical-Layer Security", "Eavesdropping Mitigation", "Atmospheric Turbulence", "Cryptographic Security"]
}}

TEXT INPUT:
{paper}
"""

## Call to the model

In [ ]:
def table_fulfill(paper, prompt_template):
    # Completa el prompt con paper y summary usando f-string
    prompt = prompt_template.format(paper=paper)

    # Callin OpenAI API
    response = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=model,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
        temperature=temperature,
    )
    return response.choices[0].message.content

## Run the extraction over all PDFs
Each response is parsed as JSON and stored under the file name.

In [ ]:
response = {}
for p in pdf_files:
    try:
        print(f"Processing: {p.name}")

        # Extract full text from the PDF
        paper = read_single_pdf(p.as_posix())

        # Call table_fulfill function
        table = table_fulfill(paper, prompt_template)

        # # Convert table from string to dictionary if needed
        # if isinstance(table, str):
        #     table = eval(table)  # Convert string to dictionary (ensure safety in production)
        # Ensure the response is a string
        if isinstance(table, str):
            # Remove code block markers if they exist
            table = table.strip("```")  # Remove triple backticks if present
            table = table.replace("python\n", "", 1)  # Remove "python\n" if it's at the start

            try:
                # Convert the cleaned string into a dictionary
                table = json.loads(table)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
                table = {}  # Return an empty dictionary in case of failure    
        # Store the result with the file name as the key
        response[p.stem] = table
    except Exception as e:
        print(f"\tError in record {p.name}: {e}")
    else:
        print(f"\tRecord {p.name} done with {len(table)} values.")

In [ ]:
response

## Merge with the first batch and save
The first 125 documents were processed in an earlier run (`df_combined.pkl`); this run adds files 126–146.

In [ ]:
df_combined = pd.read_pickle("df_combined.pkl")
df_combined.sample(8)

In [ ]:
# Transform the dictionary `response` into a DataFrame
df_response = pd.DataFrame(response)
# Transform the dictionary `response` into a DataFrame without transposition
df_response = pd.DataFrame(response).T
df_response

In [ ]:
# Append the `df_response` DataFrame to `df_combined`
df_combined = pd.concat([df_combined, df_response], ignore_index=False)
df_combined

In [ ]:
df_combined.loc['142']

In [ ]:
# Save the combined DataFrame to a pickle file
df_combined.to_pickle("df_combined2.pkl")

In [ ]:
df_combined.info()